In [1]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader

In [ ]:
# ── Model ─────────────────────────────────────────────────────────
class FraudLSTM(nn.Module):
    # Define a custom LSTM neural network for fraud detection
    
    def __init__(self, input_dim=12, hidden_dim=64, num_layers=2,
                 dropout=0.3):
        # Initialize the model with hyperparameters
        # input_dim: 12 features per transaction
        # hidden_dim: 64 neurons in LSTM hidden state
        # num_layers: 2 stacked LSTM layers
        # dropout: 0.3 (30%) to prevent overfitting
        
        super().__init__()
        # Call parent nn.Module constructor
        
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=dropout)
        # Create LSTM layer: input 12 features → 64 hidden units, 2 layers
        # batch_first=True means input shape is (batch, seq_len, features)
        # dropout applies between stacked LSTM layers
        
        self.attention = nn.Linear(hidden_dim, 1)
        # Linear layer to compute attention weights (64 → 1 score per time step)
        
        self.classifier = nn.Sequential(
            # Build classification head with 3 components:
            
            nn.Linear(hidden_dim, 32),
            # Dense layer reducing from 64 hidden units to 32
            
            nn.ReLU(),
            # Activation function to introduce non-linearity
            
            nn.Dropout(dropout),
            # Randomly zero 30% of neurons during training to prevent overfitting
            
            nn.Linear(32, 1),
            # Final dense layer: 32 → 1 output (fraud probability)
            
            nn.Sigmoid()
            # Sigmoid activation outputs value between 0 and 1 for probability
        )

    def forward(self, x):
        # Forward pass through the model
        # x: (batch, seq_len=30, features=12) - a batch of 30-step transaction histories
        
        lstm_out, _ = self.lstm(x)
        # Pass through LSTM: outputs shape (B, T=30, H=64)
        # Returns lstm_out (all time steps) and hidden state (discarded with _)
        
        attn_w = torch.softmax(
            self.attention(lstm_out), dim=1)
        # Compute attention: linear(64) → raw scores, then softmax across time steps
        # Result shape: (B, T=30, 1) - normalized weights summing to 1 across time
        
        context = (attn_w * lstm_out).sum(1)
        # Multiply each time step's output by its attention weight and sum
        # Result shape: (B, 64) - weighted combination of all time steps
        
        return self.classifier(context)
        # Pass attention-weighted context through classifier
        # Output shape: (B, 1) - fraud probability for each transaction

# ── Feature engineering (12 features per transaction) ─────────────
# amount_log, hour_sin, hour_cos, day_sin, day_cos,
# merchant_cat_enc, is_online, is_international,
# velocity_1h, velocity_24h, avg_amount_7d, days_since_last_txn

# ── Training loop (simplified) ────────────────────────────────────
def train(model, loader, epochs=10, lr=1e-3, pos_weight=50.0):
    # Training function
    # model: the FraudLSTM to train
    # loader: DataLoader with batches of (X, y)
    # epochs: number of full passes through dataset
    # lr: learning rate (0.001) controlling step size for weight updates
    # pos_weight: 50.0 penalizes false negatives 50x more (handles 0.2% fraud rate)
    
    """pos_weight handles class imbalance (~0.2% fraud rate)"""
    
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([pos_weight]))
    # Loss function combining sigmoid + binary cross-entropy
    # pos_weight=50 means: missing 1 fraud costs 50x more than false alarm
    
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    # Adam optimizer with learning rate 0.001 to update all model parameters
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, patience=2)
    # Learning rate scheduler: reduce lr if loss doesn't improve for 2 epochs
    
    for epoch in range(epochs):
        # Loop through 10 epochs (full passes through entire dataset)
        
        for X, y in loader:
            # Loop through each batch (32 samples per batch)
            # X: transaction sequences (32, 30, 12)
            # y: fraud labels (32,) - 0 or 1
            
            opt.zero_grad()
            # Clear accumulated gradients from previous batches
            
            preds = model(X).squeeze()
            # Run forward pass: get predictions, remove extra dimensions
            # preds shape: (32,)
            
            loss = criterion(preds, y.float())
            # Calculate loss between predictions and true labels
            
            loss.backward()
            # Compute gradients by backpropagating loss through network
            
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            # Clip gradients to max norm of 1.0 to prevent exploding gradients
            
            opt.step()
            # Update weights using computed gradients
            
        scheduler.step(loss)
        # Adjust learning rate based on loss (reduce if not improving)

# ── Inference with threshold calibration ──────────────────────────
def score_transaction(model, txn_sequence, threshold=0.15):
    # Inference function to score a single transaction sequence
    # model: trained FraudLSTM
    # txn_sequence: one transaction history (30, 12)
    # threshold: decision boundary (0.15) - score > 0.15 = fraud
    
    """Lower threshold = higher recall (catch more fraud)"""
    
    model.eval()
    # Switch to evaluation mode (disable dropout, batch norm)
    
    with torch.no_grad():
        # Disable gradient computation (not needed for inference, saves memory)
        
        score = model(txn_sequence.unsqueeze(0)).item()
        # Add batch dimension: (30, 12) → (1, 30, 12)
        # Run forward pass and extract scalar probability value
    
    return {
        # Return dictionary with 3 fraud signals:
        
        "fraud_score": round(score, 4),
        # Fraud probability rounded to 4 decimals (0.2547)
        
        "flag": score > threshold,
        # Boolean: True if fraud_score > 0.15 (fraud alert)
        
        "risk_tier": "HIGH" if score > 0.7
                     else "MEDIUM" if score > 0.15
                     else "LOW"
        # Risk classification: HIGH (70%+), MEDIUM (15-70%), LOW (<15%)
    }


In [ ]:
# Create dummy dataset for training
class DummyFraudDataset(Dataset):
    # Custom dataset class for fraud detection
    
    def __init__(self, num_samples=1000, seq_len=30, input_dim=12):
        # Initialize with 1000 synthetic transaction sequences
        # num_samples: 1000 total transactions
        # seq_len: 30 time steps (transaction history depth)
        # input_dim: 12 features per transaction
        
        self.data = torch.randn(num_samples, seq_len, input_dim)
        # Create random tensor of shape (1000, 30, 12) with random values
        
        self.labels = torch.randint(0, 2, (num_samples,))
        # Create random binary labels: 0 (legitimate) or 1 (fraud)

    def __len__(self):
        # Return total number of samples (1000)
        return len(self.data)

    def __getitem__(self, idx):
        # Return one sample and its label at index idx
        return self.data[idx], self.labels[idx]

# Instantiate dataset and loader
dataset = DummyFraudDataset()
# Create dataset with 1000 random samples

loader = DataLoader(dataset, batch_size=32, shuffle=True)
# Create DataLoader: generates batches of 32 samples, shuffled randomly each epoch

# Instantiate model
model = FraudLSTM()
# Create FraudLSTM with default hyperparameters

# Train the model
train(model, loader, epochs=5, lr=1e-3, pos_weight=50.0)
# Train for 5 epochs with learning rate 0.001 and class weight 50

# Example prediction: Create a sample transaction sequence
sample_txn = torch.randn(30, 12)
# Create random transaction history (30 steps, 12 features)

result = score_transaction(model, sample_txn, threshold=0.15)
# Score the transaction; flag as fraud if score > 0.15

print(result)
# Print fraud detection result dictionary


c:\Users\kotha\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\optim\lr_scheduler.py:1694: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  current = float(metrics)


{'fraud_score': 1.0, 'flag': True, 'risk_tier': 'HIGH'}
